# DMET-SSTKG — Visual-to-Graph Node-Extraction Audit

This notebook trains and evaluates the YOLOv8-based role-level detector used to generate node inputs for S-STKG construction.

The evaluation reports the visual-to-graph node-extraction performance described in the manuscript.

## Dataset sources and access

This notebook does not download or redistribute the original surgical data.
Obtain and use the required resources under the access, license, citation, and
redistribution conditions stated by their providers.

| Resource | Official information and access | Use in this notebook |
|---|---|---|
| **Cholec80** | [CAMMA dataset page](https://camma.unistra.fr/datasets/) · [Cholec80 access-request form](https://docs.google.com/forms/d/1GwZFM3-GhEduBs1d5QzbfFksKmS1OqXZAz8keYi-wKI) | Provider-authorized Cholec80 material contributing to the locally prepared combined detector dataset |
| **m2cai16-tool** | [CAMMA dataset page](https://camma.unistra.fr/datasets/) | Base M2CAI surgical tool-detection release |
| **m2cai16-tool-locations** | [Official Stanford dataset page](https://ai.stanford.edu/~syyeung/tooldetection.html) · [data archive](https://ai.stanford.edu/~syyeung/resources/m2cai16-tool-locations.zip) | Spatial tool bounding-box annotations used for Stage-1 detector initialization |

`m2cai16-tool-locations` is the spatial-annotation extension of
`m2cai16-tool`; it is not the `m2cai16-workflow` release used later for
phase-aligned S-STKG construction.

`M2CAI_TOOL_YAML` must point to the local YOLO-format configuration prepared
from `m2cai16-tool-locations`.

`COMBINED_DETECTOR_DATASET` is a locally prepared YOLO-format input rather than
a separately downloadable provider release. It must already contain the
authorized images and detector annotations required by the study. This
notebook copies that dataset, maps its nine source classes to the eight Role8
classes, and then trains the final node detector; it does not download the raw
datasets or create the original bounding-box annotations.

Dataset citations:

- A. P. Twinanda et al., “EndoNet: A Deep Architecture for Recognition Tasks
  on Laparoscopic Videos,” *IEEE Transactions on Medical Imaging*, 2017.
- A. Jin et al., “Tool Detection and Operative Skill Assessment in Surgical
  Videos Using Region-Based Convolutional Neural Networks,” *WACV*, 2018.


## 1. Required Environment

The original detector run used:

- Python 3.9
- Ultralytics 8.4.18
- PyTorch 2.4.1
- PyYAML

Install the required notebook packages before running:

```bash
pip install ultralytics==8.4.18 pyyaml
```

Use a PyTorch installation compatible with your local CPU or CUDA environment.

In [ ]:
from __future__ import annotations

import json
import shutil
from collections import Counter
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Dict

import torch
import yaml
from ultralytics import YOLO


def package_version(package_name: str) -> str:
    """Return an installed package version without failing if it is unavailable."""
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not installed"


print("Python environment")
print(f"  ultralytics: {package_version('ultralytics')}")
print(f"  torch:       {package_version('torch')}")
print(f"  pyyaml:      {package_version('pyyaml')}")


## 2. Paths and run settings

Edit only the three paths in the next cell, then run the notebook from top to bottom.

- `M2CAI_TOOL_YAML`: local M2CAI16 tool-location dataset configuration.
- `COMBINED_DETECTOR_DATASET`: local detector dataset prepared by combining the required Cholec80 and M2CAI files and annotations.
- `OUTPUT_ROOT`: directory used to store the Role8 dataset, checkpoints, and evaluation outputs.

The notebook automatically uses a CUDA device when available and otherwise uses the CPU. Existing Role8 data and trained checkpoints under `OUTPUT_ROOT` are reused; missing outputs are generated in sequence.


In [ ]:
from pathlib import Path
import torch

# Replace only these three paths with local paths.
M2CAI_TOOL_YAML = Path("/path/to/m2cai16_tool_locations/data.yaml")
COMBINED_DETECTOR_DATASET = Path("/path/to/combined_cholec80_m2cai_detector_dataset")
OUTPUT_ROOT = Path("/path/to/output_directory")

ROLE8_DATASET = OUTPUT_ROOT / "role8_detector_dataset"
ROLE8_YAML = ROLE8_DATASET / "data.yaml"
RUNS_ROOT = OUTPUT_ROOT / "yolo_role8_detector"

DEVICE = 0 if torch.cuda.is_available() else "cpu"
SEED = 0

STAGE1_RUN_NAME = "m2cai16_initialization"
STAGE2_RUN_NAME = "role8_detector"

STAGE1_WEIGHTS = RUNS_ROOT / STAGE1_RUN_NAME / "weights" / "best.pt"
FINAL_WEIGHTS = RUNS_ROOT / STAGE2_RUN_NAME / "weights" / "best.pt"

print(f"Device: {DEVICE}")
print(f"Output root: {OUTPUT_ROOT}")


## 3. Prepare the Role8 detector dataset

Copies the local combined detector dataset and maps its source classes to the eight functional–anatomical roles used by DMET-SSTKG.

In [ ]:
ROLE_NAMES = {
    0: "energy_coagulation",
    1: "clipping_control",
    2: "manipulation_traction",
    3: "dissection_energy",
    4: "irrigation_cleaning",
    5: "cutting",
    6: "specimen_handling",
    7: "anatomy_context",
}

SOURCE_TO_ROLE8 = {
    0: 0,  # Bipolar -> energy_coagulation
    1: 1,  # Clipper -> clipping_control
    2: 2,  # Grasper -> manipulation_traction
    3: 3,  # Hook -> dissection_energy
    4: 4,  # Irrigator -> irrigation_cleaning
    5: 5,  # Scissors -> cutting
    6: 6,  # Specimen_Bag -> specimen_handling
    7: 7,  # Gallbladder -> anatomy_context
    8: 7,  # Liver -> anatomy_context
}


def require_path(path: Path, description: str) -> None:
    """Raise a readable error when a required local input is missing."""
    if not path.exists():
        raise FileNotFoundError(
            f"{description} was not found: {path}\n"
            "Update the path configuration before running this cell."
        )


def write_role8_yaml(dataset_root: Path) -> Path:
    """Write the local Ultralytics dataset configuration."""
    yaml_content = {
        "path": str(dataset_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": len(ROLE_NAMES),
        "names": ROLE_NAMES,
    }

    yaml_path = dataset_root / "data.yaml"
    yaml_path.write_text(
        yaml.safe_dump(yaml_content, sort_keys=False),
        encoding="utf-8",
    )
    return yaml_path


def validate_role8_dataset(dataset_root: Path) -> Dict[int, int]:
    """Validate the expected YOLO split structure and Role8 class IDs."""
    final_counts: Counter[int] = Counter()

    for split in ("train", "val", "test"):
        image_dir = dataset_root / "images" / split
        label_dir = dataset_root / "labels" / split
        require_path(image_dir, f"Image directory for split '{split}'")
        require_path(label_dir, f"Label directory for split '{split}'")

        for label_path in sorted(label_dir.glob("*.txt")):
            for line_number, raw_line in enumerate(
                label_path.read_text(encoding="utf-8").splitlines(), start=1
            ):
                line = raw_line.strip()
                if not line:
                    continue

                fields = line.split()
                if len(fields) < 5:
                    raise ValueError(
                        f"Invalid YOLO annotation in {label_path}, "
                        f"line {line_number}: {line}"
                    )

                class_id = int(float(fields[0]))
                if class_id not in ROLE_NAMES:
                    raise RuntimeError(
                        f"Invalid Role8 class ID {class_id} found in {label_path}."
                    )
                final_counts[class_id] += 1

    return dict(sorted(final_counts.items()))


def copy_and_remap_to_role8(
    source_dataset: Path,
    output_dataset: Path,
) -> Dict[str, object]:
    """Copy a YOLO dataset and remap its class IDs to the Role8 vocabulary."""
    require_path(source_dataset, "Combined Cholec80–M2CAI detector dataset")

    if output_dataset.exists():
        raise FileExistsError(
            f"Output dataset already exists: {output_dataset}"
        )

    output_dataset.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_dataset, output_dataset)

    before_counts: Counter[int] = Counter()
    after_counts: Counter[int] = Counter()
    changed_files = 0
    changed_annotations = 0

    for split in ("train", "val", "test"):
        label_dir = output_dataset / "labels" / split
        require_path(label_dir, f"Label directory for split '{split}'")

        for label_path in sorted(label_dir.glob("*.txt")):
            original_lines = label_path.read_text(encoding="utf-8").splitlines()
            remapped_lines = []
            file_changed = False

            for line_number, raw_line in enumerate(original_lines, start=1):
                line = raw_line.strip()
                if not line:
                    continue

                fields = line.split()
                if len(fields) < 5:
                    raise ValueError(
                        f"Invalid YOLO annotation in {label_path}, line {line_number}: {line}"
                    )

                old_class = int(float(fields[0]))
                if old_class not in SOURCE_TO_ROLE8:
                    raise ValueError(
                        f"Unexpected class ID {old_class} in {label_path}, "
                        f"line {line_number}."
                    )

                new_class = SOURCE_TO_ROLE8[old_class]
                before_counts[old_class] += 1
                after_counts[new_class] += 1

                if new_class != old_class:
                    file_changed = True
                    changed_annotations += 1

                fields[0] = str(new_class)
                remapped_lines.append(" ".join(fields))

            if file_changed:
                changed_files += 1

            label_path.write_text(
                "\n".join(remapped_lines) + ("\n" if remapped_lines else ""),
                encoding="utf-8",
            )

    yaml_path = write_role8_yaml(output_dataset)
    final_counts = validate_role8_dataset(output_dataset)

    return {
        "source_dataset": str(source_dataset),
        "output_dataset": str(output_dataset),
        "yaml_path": str(yaml_path),
        "before_counts": dict(sorted(before_counts.items())),
        "after_counts": dict(sorted(after_counts.items())),
        "final_counts": final_counts,
        "changed_files": changed_files,
        "changed_annotations": changed_annotations,
    }


In [ ]:
if ROLE8_DATASET.exists():
    ROLE8_YAML = write_role8_yaml(ROLE8_DATASET)
    final_counts = validate_role8_dataset(ROLE8_DATASET)
    print(f"Using existing Role8 dataset: {ROLE8_DATASET}")
    print(f"Validated annotation counts: {final_counts}")
else:
    preparation_summary = copy_and_remap_to_role8(
        source_dataset=COMBINED_DETECTOR_DATASET,
        output_dataset=ROLE8_DATASET,
    )
    print(json.dumps(preparation_summary, indent=2))


## 4. Stage 1: M2CAI16 detector initialization

Trains a YOLOv8-Nano detector on the local M2CAI16 tool-location dataset. When the expected Stage 1 checkpoint already exists under `OUTPUT_ROOT`, the notebook reuses it instead of retraining.


In [ ]:
if STAGE1_WEIGHTS.exists():
    print(f"Using existing Stage 1 checkpoint: {STAGE1_WEIGHTS}")
else:
    require_path(M2CAI_TOOL_YAML, "M2CAI16 tool-location data.yaml")
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)

    stage1_model = YOLO("yolov8n.pt")
    stage1_model.train(
        data=str(M2CAI_TOOL_YAML),
        epochs=80,
        imgsz=640,
        batch=16,
        lr0=2e-4,
        lrf=1e-2,
        patience=20,
        save_period=20,
        warmup_epochs=5,
        weight_decay=5e-4,
        close_mosaic=20,
        seed=SEED,
        deterministic=True,
        device=DEVICE,
        project=str(RUNS_ROOT),
        name=STAGE1_RUN_NAME,
        exist_ok=True,
    )

require_path(STAGE1_WEIGHTS, "Stage 1 best checkpoint")


## 5. Role-Level Node-Extraction Model Fine-Tuning

Fine-tunes the Stage 1 checkpoint on the eight-role dataset used for S-STKG node extraction. When the expected final checkpoint already exists, the notebook reuses it instead of retraining.


In [ ]:
if FINAL_WEIGHTS.exists():
    print(f"Using existing final Role8 checkpoint: {FINAL_WEIGHTS}")
else:
    require_path(STAGE1_WEIGHTS, "Stage 1 best checkpoint")
    require_path(ROLE8_YAML, "Role8 data.yaml")
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)

    role8_model = YOLO(str(STAGE1_WEIGHTS))
    role8_model.train(
        data=str(ROLE8_YAML),
        epochs=120,
        imgsz=640,
        batch=16,
        lr0=1.5e-4,
        lrf=0.05,
        warmup_epochs=5,
        optimizer="AdamW",
        freeze=15,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.3,
        degrees=5.0,
        fliplr=0.5,
        flipud=0.0,
        mosaic=0.5,
        close_mosaic=20,
        box=7.5,
        cls=1.4,
        patience=25,
        save_period=20,
        seed=SEED,
        deterministic=True,
        device=DEVICE,
        project=str(RUNS_ROOT),
        name=STAGE2_RUN_NAME,
        exist_ok=True,
    )

require_path(FINAL_WEIGHTS, "Final Role8 detector checkpoint")


## 6. Visual-to-Graph Node-Extraction Audit

Evaluates the final Role8 checkpoint on the test split and reports the node-extraction metrics used in the manuscript.

In [ ]:
require_path(FINAL_WEIGHTS, "Final Role8 detector checkpoint")
require_path(ROLE8_YAML, "Role8 data.yaml")

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
evaluation_model = YOLO(str(FINAL_WEIGHTS))
test_metrics = evaluation_model.val(
    data=str(ROLE8_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=str(RUNS_ROOT),
    name="role8_test_evaluation",
    exist_ok=True,
)

print("Test metrics")
for metric_name, metric_value in test_metrics.results_dict.items():
    print(f"  {metric_name}: {metric_value}")


## 7. Example Node-Extraction Inference

Runs the final detector on the same test frame used by the original implementation when that frame is present. Otherwise, it automatically selects the first available test image. No additional path is required.


In [ ]:
preferred_example = (
    ROLE8_DATASET
    / "images"
    / "test"
    / "organ__video25_frame_193_endo.png"
)

if preferred_example.exists():
    EXAMPLE_IMAGE = preferred_example
else:
    test_image_dir = ROLE8_DATASET / "images" / "test"
    require_path(test_image_dir, "Role8 test-image directory")
    image_candidates = sorted(
        path
        for pattern in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff")
        for path in test_image_dir.glob(pattern)
    )
    if not image_candidates:
        raise FileNotFoundError(f"No test image was found in: {test_image_dir}")
    EXAMPLE_IMAGE = image_candidates[0]

OVERLAY_OUTPUT = OUTPUT_ROOT / "role8_detector_overlay.png"
require_path(FINAL_WEIGHTS, "Final Role8 detector checkpoint")
OVERLAY_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

inference_model = YOLO(str(FINAL_WEIGHTS))
prediction = inference_model.predict(
    source=str(EXAMPLE_IMAGE),
    imgsz=640,
    conf=0.4,
    save=False,
    verbose=False,
    line_width=12,
    device=DEVICE,
)

prediction[0].save(filename=str(OVERLAY_OUTPUT))
print(f"Example image: {EXAMPLE_IMAGE}")
print(f"Saved detector overlay to: {OVERLAY_OUTPUT}")


## Output

The final Role8 checkpoint is stored under:

```text
<OUTPUT_ROOT>/yolo_role8_detector/role8_detector/weights/best.pt
```

The test evaluation is stored under the same `yolo_role8_detector` directory, and the qualitative overlay is saved as:

```text
<OUTPUT_ROOT>/role8_detector_overlay.png
```

This final checkpoint is used by the following S-STKG construction stage.
